In [1]:
import json
import re
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

In [2]:
FAMILY_MAP = {
    "Aire acondicionado": "AA",
    "Estacion de Carga Portatil": "GENKI",
    "TPMS": "TPMS",
    "TPMS Repuesto": "TPMS",
    "Climatizador": "CLIMATIZADOR",
    "Carjack": "CARJACK",
    "Arrancador": "CARJACK",
    "Inflador": "CARJACK",
    "Caldera": "CALDERA",
}

In [3]:

def normalize_sku(x) -> str:
    return str(x).strip()

def split_images(s: str) -> list[str]:
    if not isinstance(s, str) or not s.strip():
        return []
    return [u.strip() for u in s.split(",") if u.strip()]

def to_bool(x):
    if pd.isna(x):
        return None
    return bool(x)

In [4]:
def export_from_excel(
    xlsx_path: Path,
    out_path: Path,
    sheet_prices="00 _Precios y links",
    sheet_input="Input",
    sheet_sku_master="Datos",
):
    df_prices = pd.read_excel(xlsx_path, sheet_name=sheet_prices)
    df_input = pd.read_excel(xlsx_path, sheet_name=sheet_input)
    df_master = pd.read_excel(xlsx_path, sheet_name=sheet_sku_master)

    # keep meaningful price rows (your sheet has many blank formatting rows)
    prices = df_prices.dropna(
        subset=["SKU", "tipo de producto / Familia", "Modelo"]).copy()
    prices = prices[prices["SKU"].astype(str).str.strip().ne("0")]

    # pick canonical product row per SKU (Variante Principal == 1)
    canon = df_input[df_input["Variante Principal"] == 1].copy()

    merged = prices.merge(
        canon[["SKU", "ID", "Nombre", "Categorias",
               "Imagenes", "Activo", "Publicado", "Comprable"]],
        on="SKU",
        how="left",
    )

    items = []
    priced_skus = set()

    for _, r in merged.iterrows():
        sku = normalize_sku(r["SKU"])
        priced_skus.add(sku)

        fam_raw = str(r["tipo de producto / Familia"]).strip()
        fam = FAMILY_MAP.get(fam_raw, "OTHER")

        items.append({
            "sku": sku,
            "family": fam,
            "family_raw": fam_raw,
            "model": str(r["Modelo"]).strip(),
            "name": str(r["Nombre"]).strip() if pd.notna(r["Nombre"]) else None,
            "price": {"ars": float(r["precio_ars"]) if pd.notna(r["precio_ars"]) else None, "usd": None},
            "availability": {
                "active": to_bool(r.get("Activo")),
                "published": to_bool(r.get("Publicado")),
                "purchasable": to_bool(r.get("Comprable")),
            },
            "ids": {"product_id": str(r["ID"]).strip() if pd.notna(r["ID"]) else None},
            "taxonomy": {"categories": str(r["Categorias"]).strip() if pd.notna(r["Categorias"]) else None},
            "media": {"images": split_images(r.get("Imagenes"))},
            "links": {
                "product_page": None,
                "support_url": None,
                "sales_url": None,
                "whatsapp_support": None,
                "whatsapp_sales": None
            },
            "aliases": sorted(set(filter(None, [
                sku,
                str(r["Modelo"]).strip() if pd.notna(r["Modelo"]) else None,
                str(r["Nombre"]).strip() if pd.notna(r["Nombre"]) else None
            ]))),
        })

    # add master SKUs with missing prices (kept as null to avoid hallucinations)
    for _, r in df_master.iterrows():
        sku = normalize_sku(r["SKU"])
        if sku in priced_skus:
            continue
        fam_raw = str(r.get("Tipo de producto", "")).strip()
        fam = FAMILY_MAP.get(fam_raw, "OTHER")
        model = str(r.get("Modelo", "")).strip()

        items.append({
            "sku": sku,
            "family": fam,
            "family_raw": fam_raw,
            "model": model,
            "name": None,
            "price": {"ars": None, "usd": None},
            "availability": {"active": None, "published": None, "purchasable": None},
            "ids": {"product_id": None},
            "taxonomy": {"categories": None},
            "media": {"images": []},
            "links": {
                "product_page": None,
                "support_url": None,
                "sales_url": None,
                "whatsapp_support": None,
                "whatsapp_sales": None
            },
            "aliases": sorted(set(filter(None, [sku, model]))),
        })

    catalog = {
        "schema_version": "catalog.v1",
        "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "defaults": {"currency": "ARS", "country": "AR"},
        "source": {
            "type": "excel",
            "file": xlsx_path.name,
            "sheets": {
                "prices": sheet_prices,
                "enrichment": sheet_input,
                "sku_master": sheet_sku_master,
            },
        },
        "items": sorted(items, key=lambda x: x["sku"]),
    }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(
        catalog, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Wrote {len(catalog['items'])} items to {out_path}")

In [12]:
# Path to .xlsx or .pdf source
in_path = Path("../knowledge/00 _Precios y links editable_V02.xlsx")
out_path = Path("../data/catalog.json")  # Output catalog.json path

if in_path.suffix.lower() in [".xlsx", ".xls"]:
    export_from_excel(in_path, out_path)
# elif in_path.suffix.lower() == ".pdf":
#     export_from_pdf(in_path, out_path)
else:
    raise SystemExit("Unsupported input type. Use .xlsx or .pdf")

Wrote 71 items to ../data/catalog.json


In [6]:
# import camelot

# def export_from_pdf(pdf_path: Path, out_path: Path):
#     """
#     Minimal PDF support. Expects a table with columns like:
#     SKU | tipo de producto / Familia | Modelo | precio_ars

#     Requires: uv add camelot-py
#     Note: PDF table extraction can be brittle; Excel is strongly preferred.
#     """

#     tables = camelot.read_pdf(str(pdf_path), pages="all", flavor="stream")
#     if tables.n == 0:
#         raise SystemExit("No tables detected in PDF.")

#     # naive: take first table and rename columns
#     df = tables[0].df
#     df.columns = df.iloc[0]
#     df = df.iloc[1:].copy()

#     # try to match required columns
#     required = ["SKU", "tipo de producto / Familia", "Modelo", "precio_ars"]
#     missing = [c for c in required if c not in df.columns]
#     if missing:
#         raise SystemExit(f"PDF table missing columns: {missing}")

#     # build a minimal catalog (no enrichment)
#     items = []
#     for _, r in df.iterrows():
#         sku = normalize_sku(r["SKU"])
#         fam_raw = str(r["tipo de producto / Familia"]).strip()
#         fam = FAMILY_MAP.get(fam_raw, "OTHER")

#         # parse price
#         p = str(r["precio_ars"]).replace(".", "").replace(",", ".")
#         try:
#             price_ars = float(p)
#         except:
#             price_ars = None

#         items.append({
#             "sku": sku,
#             "family": fam,
#             "family_raw": fam_raw,
#             "model": str(r["Modelo"]).strip(),
#             "name": None,
#             "price": {"ars": price_ars, "usd": None},
#             "availability": {"active": None, "published": None, "purchasable": None},
#             "ids": {"product_id": None},
#             "taxonomy": {"categories": None},
#             "media": {"images": []},
#             "links": {"product_page": None, "support_url": None, "sales_url": None, "whatsapp_support": None, "whatsapp_sales": None},
#             "aliases": sorted(set(filter(None, [sku, str(r["Modelo"]).strip()]))),
#         })

#     catalog = {
#         "schema_version": "catalog.v1",
#         "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
#         "defaults": {"currency": "ARS", "country": "AR"},
#         "source": {"type": "pdf", "file": pdf_path.name, "sheets": {}},
#         "items": sorted(items, key=lambda x: x["sku"]),
#     }

#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     out_path.write_text(json.dumps(
#         catalog, ensure_ascii=False, indent=2), encoding="utf-8")
#     print(f"Wrote {len(catalog['items'])} items to {out_path}")